# 08 — TinyStories: a ~25M model that writes coherent English (Colab / GPU)

Everything so far ran on a laptop with a char-level model on Shakespeare — great for
*understanding*, but the output is only Shakespeare-*shaped*. This notebook is the **demo**: a
15–30M-parameter model trained with our **from-scratch BPE tokenizer** on **TinyStories**, a
dataset of short children's stories written with a small vocabulary. The famous result (Eldan &
Li, 2023) is that even *tiny* models trained on TinyStories produce **grammatical, coherent
English** — perfect for a portfolio demo.

> **Run this on Colab (or any CUDA GPU), not the laptop.** It downloads ~2 GB and needs a GPU to
> train in a reasonable time (≈1–2 hours on a free T4). Set **Runtime → Change runtime type →
> T4 GPU**. The cells are ordered to run top to bottom.

The model and training code are the *exact same* `model.py` / `train.py` from the earlier
notebooks — only the tokenizer (BPE now), data, and model size change. That's the whole point:
what you understood at small scale runs unchanged at 30× the size.

In [1]:
# --- Colab setup: clone the repo (or skip if you've mounted it) -----------------
# !git clone https://github.com/<you>/transformer-from-scratch.git
# %cd transformer-from-scratch
!pip -q install tiktoken regex
import torch
assert torch.cuda.is_available(), "Enable a GPU: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

AssertionError: Enable a GPU: Runtime -> Change runtime type -> T4 GPU

In [ ]:
import sys, os, math, time, pickle
from pathlib import Path
import numpy as np
sys.path.insert(0, ".")           # repo root on Colab
from model import GPT, GPTConfig
from bpe import BPETokenizer
device = "cuda"
torch.manual_seed(1337)

## 1. Get TinyStories

We pull the raw text files from the Hugging Face dataset mirror (just text, we're not using the
`datasets` library or any model code from HF). We use the validation split as our held-out set.

In [ ]:
os.makedirs("data/tinystories", exist_ok=True)
base = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main"
for fname in ["TinyStoriesV2-GPT4-train.txt", "TinyStoriesV2-GPT4-valid.txt"]:
    dst = f"data/tinystories/{fname}"
    if not os.path.exists(dst):
        print("downloading", fname, "...")
        os.system(f"wget -q -O {dst} {base}/{fname}")
    print(fname, f"{os.path.getsize(dst)/1e6:.0f} MB")

## 2. Train our BPE tokenizer on the corpus

We reuse the from-scratch `BPETokenizer` from notebook 01. Training a full-vocab tokenizer on 2GB
of Python is slow, so we (a) train the merges on a representative **subset** of the text — merge
frequencies converge fast, a few hundred MB is plenty — and (b) pick a **4,096-token** vocab: big
enough that common words are single tokens, small enough to keep the embedding matrix modest for a
25M model. This is the credibility centerpiece: **our own BPE, no HF tokenizer.**

In [ ]:
VOCAB_SIZE = 4096
tok_path = "data/tinystories/bpe_4096.json"

if os.path.exists(tok_path):
    tok = BPETokenizer.load(tok_path)
else:
    # train merges on the first ~200MB (enough for stable merge statistics)
    with open("data/tinystories/TinyStoriesV2-GPT4-train.txt") as f:
        sample_text = f.read(200_000_000)
    t0 = time.time()
    tok = BPETokenizer()
    tok.train(sample_text, vocab_size=VOCAB_SIZE, verbose=True)
    tok.save(tok_path)
    print(f"trained {VOCAB_SIZE}-token BPE in {(time.time()-t0)/60:.1f} min")
    del sample_text

demo = "Once upon a time, there was a little robot who loved to paint."
print("\nexample:", "|".join(tok.vocab[i].decode('utf-8','replace') for i in tok.encode(demo)))
print(f"chars/token on the demo: {len(demo)/len(tok.encode(demo)):.2f}")

## 3. Tokenize the corpus to `.bin` files

We encode the whole dataset once and memmap it during training — same `train.bin`/`val.bin`/
`meta.pkl` layout as the Shakespeare run, so `train.py` doesn't change. Our pure-Python `encode`
isn't fast, so we cache encoded chunks and process in a stream. (This cell is the slow one —
~10–20 min. It only runs once; the `.bin` files are reused every training run.)

In [ ]:
def encode_file_to_bin(txt_path, bin_path, tok, flush_every=100_000):
    '''Stream the file, encode chunk-by-chunk with a cache, append to the .bin
       in blocks. Streaming (not read()) keeps memory flat on a 2GB file; the
       cache exploits TinyStories' heavy repetition to make pure-Python encode
       fast enough.'''
    if os.path.exists(bin_path):
        print(bin_path, "exists, skipping"); return
    from functools import lru_cache
    import regex
    pat = regex.compile(tok.pattern)
    enc_chunk = lru_cache(maxsize=200_000)(lambda c: tuple(tok._encode_chunk(c.encode("utf-8"))))
    total = 0
    with open(txt_path) as f, open(bin_path, "wb") as out:
        buf = []
        for line in f:                       # stream line by line, constant memory
            for chunk in pat.findall(line):
                buf.extend(enc_chunk(chunk))
            if len(buf) >= flush_every:
                np.array(buf, dtype=np.uint16).tofile(out); total += len(buf); buf = []
        if buf:
            np.array(buf, dtype=np.uint16).tofile(out); total += len(buf)
    print(f"{bin_path}: {total:,} tokens")

encode_file_to_bin("data/tinystories/TinyStoriesV2-GPT4-valid.txt", "data/tinystories/val.bin", tok)
encode_file_to_bin("data/tinystories/TinyStoriesV2-GPT4-train.txt", "data/tinystories/train.bin", tok)

pickle.dump({"vocab_size": tok.vocab_size, "kind": "bpe",
             "tokenizer_path": os.path.abspath(tok_path)},
            open("data/tinystories/meta.pkl", "wb"))
print("wrote meta.pkl, vocab_size =", tok.vocab_size)

## 4. The model: ~25M parameters

We scale up the same `GPT`: **6 layers, 6 heads, 384-dim, 512 context**. With a 4,096 vocab that's
about 25M parameters — right in the sweet spot the TinyStories paper showed produces coherent
text. Compare to the 0.8M Shakespeare model: same architecture, ~30× the capacity.

In [ ]:
config = GPTConfig(vocab_size=tok.vocab_size, block_size=512,
                   n_layer=6, n_head=6, n_embd=384, dropout=0.0)
model = GPT(config).to(device)
print(f"parameters: {model.num_params()/1e6:.1f}M")

## 5. Train

Same loop as notebook 04 / `train.py`, with settings tuned for the bigger job:

- **mixed precision** (bf16 if the GPU supports it, else fp16 + GradScaler) — active now that
  we're on a real GPU,
- **gradient accumulation** to reach a large effective batch,
- more steps (~5000) since there's vastly more data.

You can also just run the script: `!python train.py --dataset tinystories --n_layer 6 --n_head 6
--n_embd 384 --block_size 512 --batch_size 32 --grad_accum_steps 4 --max_iters 5000
--learning_rate 6e-4 --dropout 0.0`. We inline it here so the notebook is self-contained.

In [ ]:
from contextlib import nullcontext
train_data = np.memmap("data/tinystories/train.bin", dtype=np.uint16, mode="r")
val_data   = np.memmap("data/tinystories/val.bin",   dtype=np.uint16, mode="r")

BLOCK, BATCH, ACCUM = 512, 32, 4
MAX_ITERS, WARMUP, EVAL_EVERY, EVAL_ITERS = 5000, 200, 500, 100
LR_MAX, LR_MIN = 6e-4, 6e-5

def get_batch(split):
    d = train_data if split=="train" else val_data
    ix = torch.randint(len(d)-BLOCK-1, (BATCH,))
    x = torch.stack([torch.from_numpy(d[i:i+BLOCK].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(d[i+1:i+1+BLOCK].astype(np.int64)) for i in ix])
    return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)

amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
autocast = torch.autocast("cuda", dtype=amp_dtype)
scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype==torch.float16)

decay   = [p for p in model.parameters() if p.dim() >= 2]
nodecay = [p for p in model.parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{"params":decay,"weight_decay":0.1},{"params":nodecay,"weight_decay":0.0}],
                        lr=LR_MAX, betas=(0.9,0.95))
def get_lr(it):
    if it < WARMUP: return LR_MAX*(it+1)/WARMUP
    r=(it-WARMUP)/(MAX_ITERS-WARMUP); return LR_MIN+0.5*(1+math.cos(math.pi*r))*(LR_MAX-LR_MIN)

@torch.no_grad()
def est_loss():
    model.eval(); out={}
    for sp in ("train","val"):
        L=torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            x,y=get_batch(sp)
            with autocast: _,l=model(x,y)
            L[k]=l.item()
        out[sp]=L.mean().item()
    model.train(); return out

In [ ]:
history, best = [], float("inf"); t0=time.time(); model.train()
for it in range(MAX_ITERS+1):
    for g in opt.param_groups: g["lr"]=get_lr(it)
    if it % EVAL_EVERY == 0 or it==MAX_ITERS:
        ls=est_loss(); history.append((it,ls["train"],ls["val"]))
        print(f"iter {it:5d} | train {ls['train']:.3f} | val {ls['val']:.3f} | {(time.time()-t0)/60:.1f} min")
        if ls["val"]<best:
            best=ls["val"]; os.makedirs("checkpoints",exist_ok=True)
            torch.save({"model":model.state_dict(),"config":config,
                        "meta":pickle.load(open('data/tinystories/meta.pkl','rb')),
                        "iter":it,"val_loss":best,"history":history},
                       "checkpoints/tinystories.pt")
    if it==MAX_ITERS: break
    opt.zero_grad(set_to_none=True)
    for _ in range(ACCUM):
        x,y=get_batch("train")
        with autocast: _,l=model(x,y); l=l/ACCUM
        scaler.scale(l).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    scaler.step(opt); scaler.update()
print(f"done, best val {best:.3f}")

## 6. Generate stories

The payoff. We load the best checkpoint and sample with our from-scratch decoding (temperature +
top-p) and the KV cache from notebook 07. At ~25M params on TinyStories you should get fully
grammatical, on-topic short stories — a dramatic step up from the char-level Shakespeare model.

In [ ]:
ckpt = torch.load("checkpoints/tinystories.pt", map_location=device, weights_only=False)
m = GPT(ckpt["config"]).to(device); m.load_state_dict(ckpt["model"]); m.eval()

prompts = [
    "Once upon a time, there was a little girl named Lily.",
    "Tom and Sara found a big box in the garden.",
    "The dragon was not scary at all. In fact,",
]
for p in prompts:
    ids = torch.tensor([tok.encode(p)], device=device)
    out = m.generate(ids, max_new_tokens=200, temperature=0.8, top_p=0.9, use_cache=True)
    print(tok.decode(out[0].tolist())); print("-"*70)

In [ ]:
# plot the loss curve for the writeup
import matplotlib.pyplot as plt
its=[h[0] for h in ckpt["history"]]; tr=[h[1] for h in ckpt["history"]]; va=[h[2] for h in ckpt["history"]]
plt.figure(figsize=(8,4))
plt.plot(its,tr,"-o",ms=3,label="train"); plt.plot(its,va,"-o",ms=3,label="val")
plt.xlabel("iteration"); plt.ylabel("cross-entropy loss (BPE tokens)")
plt.title("TinyStories 25M — training curve"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig("assets/tinystories_loss.png", dpi=110, bbox_inches="tight"); plt.show()

## Takeaways

- The **same `model.py` and training loop** that fit an 0.8M char model scale, unchanged, to a
  25M BPE model on a 2GB corpus. Understanding the small case *is* understanding the large case.
- **Our from-scratch BPE** (notebook 01) does the real tokenization here — the credibility win the
  project calls for.
- TinyStories is the trick that makes a *small* model produce *coherent* English: constrain the
  data to simple vocabulary and short narratives, and 25M parameters is enough to learn grammar,
  characters, and simple plot.
- Everything the earlier notebooks built — mixed precision, grad accumulation, top-p sampling, the
  KV cache — pays off here at scale.

That's the full project: from a character lookup table to a coherent-English generator, every
component built and understood from scratch. See the [README](../README.md) for the write-up.